# **Partie 2 : Agent outillé LangChain**
### *Assistant juridique sur des textes de loi PDF*

Ce notebook présente l'ajout d'outils à un assistant intelligent.  
L'objectif ici n'est pas de faire le RAG sur les PDF, mais de construire un agent capable de choisir un outil selon la question posée.

### *Ce que fait ce notebook*
- charge l'environnement de travail local ;
- vérifie l'accès à l'API OpenAI ;
- définit les schémas d'entrée des outils ;
- implémente plusieurs outils utiles au cas d'usage juridique ;
- construit un agent LangChain ;
- teste l'agent sur quelques exemples ;
- ajoute un mini routeur préparatoire à l'intégration finale avec le RAG.

### *Outils présents*
- calculatrice juridique ;
- météo ;
- recherche web juridique ;
- date du jour ;
- todo locale.

## **1. Préparation de l'environnement notebook**

On va :
- ajouter la racine du projet au `sys.path` ;
- recharger proprement les variables d'environnement ;
- vérifier que la clé OpenAI utilisée par le notebook est bien celle attendue ;
- tester ensuite l'accès à l'API avant de construire l'agent.

### **Imports techniques**

On importe :
- les bibliothèques standard Python ;
- les dépendances de configuration ;
- Pydantic pour typer les entrées des outils ;
- LangChain pour l'agent et les tools.

In [136]:
import os
import re
import json
import requests
import sys

from datetime import datetime
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.agents import create_agent
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings


In [137]:
load_dotenv(Path("../.env"), override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

print("OpenAI OK ?", bool(OPENAI_API_KEY))

OpenAI OK ? True


In [138]:
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = C:\Users\PVeena\Desktop\Generative_IA\GEN_AI_Assistant_Juridique


In [139]:
# Nettoyer toute ancienne clé
os.environ.pop("OPENAI_API_KEY", None)

# Charger le bon .env
load_dotenv(Path("../.env"), override=True)

key = os.getenv("OPENAI_API_KEY")

print("Clé chargée ?", bool(key))
print("Début :", key[:12] if key else None)
print("Fin :", key[-6:] if key else None)

Clé chargée ? True
Début : sk-proj-TGlH
Fin : NaVfcA


### **Test direct de l'accès OpenAI**

Avant d'utiliser LangChain, on teste ici le client OpenAI directement.  
Cela permet de distinguer un problème de clé / environnement d'un problème de code agentique.

In [140]:
from openai import OpenAI

client = OpenAI(api_key=key)

response = client.responses.create(
    model="gpt-4o-mini",
    input="Dis juste bonjour"
)

print(response.output_text)

Bonjour ! Comment puis-je t’aider aujourd’hui ?


### **Vérification rapide des imports critiques**

Cette cellule sert de diagnostic rapide pour vérifier que les dépendances essentielles sont bien installées dans l'environnement du notebook.

In [141]:
try:
    from dotenv import load_dotenv
    print("dotenv OK")
except Exception as e:
    print("dotenv ERREUR:", e)

try:
    from pydantic import BaseModel, Field
    print("pydantic OK")
except Exception as e:
    print("pydantic ERREUR:", e)

try:
    from langchain.agents import create_agent
    print("create_agent OK")
except Exception as e:
    print("create_agent ERREUR:", e)

try:
    from langchain_core.tools import StructuredTool
    print("StructuredTool OK")
except Exception as e:
    print("StructuredTool ERREUR:", e)

try:
    from langchain_openai import ChatOpenAI
    print("ChatOpenAI OK")
except Exception as e:
    print("ChatOpenAI ERREUR:", e)

dotenv OK
pydantic OK
create_agent OK
StructuredTool OK
ChatOpenAI OK


## **3. Chargement de la configuration**

On lit ici les clés d'API depuis l'environnement pour alimenter :
- OpenAI ;
- OpenWeather ;
- Tavily.

In [142]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

print("OpenAI OK ?", bool(OPENAI_API_KEY))
print("Weather OK ?", bool(OPENWEATHER_API_KEY))
print("Tavily OK ?", bool(TAVILY_API_KEY))

OpenAI OK ? True
Weather OK ? True
Tavily OK ? True


## **4. Définition des schémas d'entrée des outils**

Les schémas Pydantic servent à :
- clarifier les entrées attendues ;
- rendre les tools plus explicites pour l'agent ;
- améliorer la robustesse des appels.

In [143]:
class CalcInput(BaseModel):
    expression: str = Field(..., description="Expression mathématique simple")

class WeatherInput(BaseModel):
    ville: str = Field(..., description="Nom de la ville")

class WebSearchInput(BaseModel):
    question: str = Field(..., description="Question juridique récente")

class TodoInput(BaseModel):
    action: Literal["ajouter", "lister", "supprimer"]
    item: str = ""

class EmptyInput(BaseModel):
    pass

## **5. Implémentation des outils**

On construit maintenant les fonctions qui seront exposées à l'agent.
Chaque outil correspond à un besoin précis du sujet.

### **5.1 Outil de calcul juridique**

Ce premier outil gère les calculs simples utiles en contexte juridique :
- pourcentages ;
- montants ;
- additions et multiplications simples.

La première version est volontairement stricte.

In [144]:
def safe_eval_math(expression: str) -> float:
    if not re.fullmatch(r"[0-9\s\+\-\*/\(\)\.]+", expression):
        raise ValueError("Expression non autorisée")

    result = eval(expression, {"__builtins__": {}}, {})

    if not isinstance(result, (int, float)):
        raise ValueError("Résultat non numérique")

    return float(result)


def calculatrice_juridique(expression: str) -> str:
    value = safe_eval_math(expression)
    return f"Résultat : {value}"

### **5.2 Outil météo**

Cet outil illustre l'appel à une API externe et répond aux questions de type :
*Quel temps fait-il à Paris aujourd'hui ?*

In [145]:
def fetch_weather(city: str):
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city,
        "appid": OPENWEATHER_API_KEY,
        "units": "metric",
        "lang": "fr",
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()


def meteo_ville(ville: str) -> str:
    if not OPENWEATHER_API_KEY:
        return "API météo non configurée"

    data = fetch_weather(ville)

    return f"Météo à {ville} : {data['weather'][0]['description']}, {data['main']['temp']}°C"

### **5.3 Outil de recherche web juridique**

Cet outil sert pour des questions d'actualité ou de veille juridique qui ne relèvent pas du corpus PDF local.

In [146]:
def tavily_search(question: str):
    url = "https://api.tavily.com/search"
    payload = {
        "api_key": TAVILY_API_KEY,
        "query": question,
        "max_results": 3,
    }
    response = requests.post(url, json=payload)
    response.raise_for_status()
    return response.json()


def recherche_web_juridique(question: str) -> str:
    if not TAVILY_API_KEY:
        return "Recherche web indisponible"

    data = tavily_search(question)

    results = data.get("results", [])

    return "\n".join([r["title"] for r in results])

### **5.4 Outil date**

Outil simple mais utile pour contextualiser certaines questions temporelles.

In [147]:
def date_du_jour() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

### **5.5 Outil todo locale**

Cette fonction simule une mini gestion de tâches pour un dossier juridique.

In [148]:
def todo_dossier_local(action: str, item: str = "") -> str:
    path = Path("todo.json")
    data = json.loads(path.read_text()) if path.exists() else []

    if action == "ajouter":
        data.append(item)
        path.write_text(json.dumps(data))
        return "Ajouté"

    if action == "lister":
        return str(data)

    if action == "supprimer":
        data.remove(item)
        path.write_text(json.dumps(data))
        return "Supprimé"

    return "Action invalide"

## **6. Enregistrement des tools dans LangChain**

On transforme ici les fonctions Python en `StructuredTool` afin qu'elles puissent être sélectionnées automatiquement par l'agent.

In [ ]:
tools = [
    StructuredTool.from_function(
        func=calculatrice_juridique,
        name="calculatrice",
        description="Effectue un calcul numérique. Accepte aussi des expressions comme '15% de 2000', des pourcentages, des montants, des intérêts et des pénalités.",
        args_schema=CalcInput
    ),
    StructuredTool.from_function(
        func=meteo_ville,
        name="meteo",
        description="Donne la météo actuelle d'une ville.",
        args_schema=WeatherInput
    ),
    StructuredTool.from_function(
        func=recherche_web_juridique,
        name="web",
        description="Recherche des informations juridiques récentes sur internet.",
        args_schema=WebSearchInput
    ),
    StructuredTool.from_function(
        func=lambda: date_du_jour(),
        name="date",
        description="Donne la date et l'heure actuelle.",
        args_schema=EmptyInput
    ),
    StructuredTool.from_function(
        func=todo_dossier_local,
        name="todo",
        description="Gère une liste de tâches pour un dossier juridique.",
        args_schema=TodoInput
    ),
]

## **7. Construction du modèle et du prompt système**

Le prompt système fixe le comportement attendu :
- utiliser les tools si nécessaire ;
- ne pas inventer de lois ;
- signaler qu'une question PDF relève du futur module RAG.

In [150]:
SYSTEM_PROMPT = """
Tu es un assistant juridique.

- utilise les tools si nécessaire
- n'invente jamais de lois
- si la question concerne des PDF -> dire que RAG sera utilisé
"""

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=key
)


### **Instanciation de l'agent**

L'agent combine maintenant :
- le modèle ;
- les tools ;
- les instructions système.

In [151]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT
)

### **Fonction utilitaire d'appel**

Cette fonction simplifie les tests ultérieurs en encapsulant l'appel à `agent.invoke`.

In [152]:
def ask_agent(query: str):
    result = agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].content

## **8. Premiers tests unitaires rapides**

On vérifie d'abord quelques fonctions simples avant de tester l'agent complet.

In [153]:
print(calculatrice_juridique("100 + 20 * 3"))
print(date_du_jour())
print(todo_dossier_local("ajouter", "vérifier contrat"))
print(todo_dossier_local("lister"))

Résultat : 160.0
2026-04-13 01:43:06
Ajouté
['vérifier contrat', 'vérifier contrat', 'vérifier contrat', 'vérifier contrat', 'vérifier contrat', 'vérifier contrat', 'vérifier contrat']


## 9. **Amélioration de la gestion des expressions mathématiques**

La première version du calculateur était trop stricte pour des formulations naturelles comme :
- `15% de 2000`
- `10 % de 500 + 20`

On ajoute donc ici une normalisation plus robuste **sans changer l'objectif de l'outil**.

In [154]:
def normalize_math_expression(expression: str) -> str:
    """
    Normalise des formulations courantes en une expression mathématique simple.
    Exemples :
    - '15% de 2000' -> '(15/100) * 2000'
    - '10 % de 500 + 20' -> '(10/100) * 500 + 20'
    """
    expr = expression.strip().lower()

    # Virgules françaises -> points
    expr = expr.replace(",", ".")

    # "15% de 2000" ou "15 % de 2000" -> "(15/100) * 2000"
    expr = re.sub(r"(\d+(?:\.\d+)?)\s*%\s*de\s*(\d+(?:\.\d+)?)", r"(\1/100) * \2", expr)

    # "15%" tout seul -> "(15/100)"
    expr = re.sub(r"(\d+(?:\.\d+)?)\s*%", r"(\1/100)", expr)

    # On retire les mots parasites fréquents
    expr = expr.replace("euros", "")
    expr = expr.replace("euro", "")
    expr = expr.replace("€", "")
    expr = expr.replace("x", "*")

    # Espaces multiples
    expr = re.sub(r"\s+", " ", expr).strip()

    return expr


def safe_eval_math(expression: str) -> float:
    expr = normalize_math_expression(expression)

    # Autoriser uniquement chiffres, espaces, parenthèses, opérateurs et points
    if not re.fullmatch(r"[0-9\s\+\-\*/\(\)\.]+", expr):
        raise ValueError(f"Expression non autorisée après normalisation : {expr}")

    result = eval(expr, {"__builtins__": {}}, {})

    if not isinstance(result, (int, float)):
        raise ValueError("Résultat non numérique")

    return float(result)


def calculatrice_juridique(expression: str) -> str:
    """
    Effectue un calcul numérique utile en contexte juridique :
    pourcentage, intérêts simples, montants, frais, pénalités.
    """
    value = safe_eval_math(expression)
    return f"Résultat du calcul : {value}"

### **Vérification de la nouvelle logique de calcul**

On teste ici plusieurs expressions plus proches du langage naturel.

In [155]:
print(calculatrice_juridique("15% de 2000"))
print(calculatrice_juridique("10 % de 500 + 20"))
print(calculatrice_juridique("(2500 * 0.15) + 120"))

Résultat du calcul : 300.0
Résultat du calcul : 70.0
Résultat du calcul : 495.0


## **10. Tests de l'agent sur des calculs**

Ces cellules vérifient que l'agent utilise bien le tool calcul après amélioration.
### **Démonstration de plusieurs cas d'usage**

On teste maintenant :
- un calcul ;
- une question météo ;
- une salutation ;
- une question qui devrait relever du futur RAG.

In [156]:
print(ask_agent("calcule 15% de 2000"))
print(ask_agent("quel temps fait-il à Paris"))
print(ask_agent("bonjour"))
print(ask_agent("selon le pdf, que dit l'article 1240 ?"))

15% de 2000 est égal à 300.
Le temps à Paris est couvert avec une température de 8,31°C.
Bonjour ! Comment puis-je vous aider aujourd'hui ?
Je vais utiliser RAG pour extraire les informations de l'article 1240 à partir du PDF. Veuillez patienter un moment.


## **11. Routeur simple préparatoire à l'intégration finale**

Ce routeur ne remplace pas l'agent.  
Il sert seulement à montrer comment on pourrait, dans la partie finale du projet, distinguer :
- une requête documentaire pour le RAG ;
- une requête outillée ;
- une conversation normale.

In [157]:
def classify_query(query: str):
    q = query.lower()

    if "pdf" in q or "article" in q:
        return "rag"
    if "météo" in q or "calcule" in q:
        return "tool"

    return "chat"

### **Vérification du routeur**

Petits exemples pour contrôler le comportement du classifieur heuristique.

In [158]:
print(classify_query("pdf article"))
print(classify_query("calcule 10"))
print(classify_query("bonjour"))

rag
tool
chat
